# Set limits and stopping conditions

Limit how many revisions a run tries, stop when its score is good enough, or
cap its evaluation work. Start with `trials` for proposal attempts; use a
`Task` when you need an objective target or an evaluation ceiling.

| You want to… | Setting |
|---|---|
| [Limit proposal attempts](#3-allow-one-proposal-attempt) | `trials` / `max_trials` |
| [Stop at a target score](#4-stop-when-the-score-is-good-enough) | `satisfy` |
| [Cap evaluations, including the seed](#5-put-a-ceiling-on-evaluation-work) | `evaluations` in `Budget` |
| [Narrow one execution](#6-narrow-the-budget-for-one-execution) | `budget` in `run()` |

The examples keep the same duration parser, proposer, and evaluator, so you
can see exactly what each control changes. The handwritten revisions score
33%, 67%, and 100% on six fixed checks. No model, SDK, or credentials are needed.



<a id="1-install"></a>
<a id="2-define-the-experiments-inputs"></a>

## Required setup for a fresh notebook

**Install.**

Use a fresh notebook environment running **Python 3.12 or newer**.
Install directly from the published documentation:

In [ ]:
%pip install https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip

If you already imported Meta-Evolve, restart the kernel after installing.
Then run the remaining cells in order.

**Archived or offline docs:** use the ZIP included with that build. Put
`meta-evolve.zip` in the notebook's working folder (`%pwd` shows it; hosted
notebooks let you upload files), then run `%pip install ./meta-evolve.zip`
instead. Installing from source may still download build tools.

The [package ZIP](https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip) is also included with this
documentation build.

**Starting source and instructions.** `SEED` reads a duration's number but
ignores its unit. `CONTRACT` tells the proposer what the parser should do.

In [ ]:
import meta_evolve as meta

CONTRACT = """Implement parse_seconds(text) for whole-number durations.
Inputs contain a number followed by s, m, or h, such as '30s' or '2h'.
Return the duration in seconds as an integer. Return only Python source.
"""

SEED = '''def parse_seconds(text):
    return int(text[:-1])
'''

**Evaluation.** `evaluate` returns the fraction of six checks answered
correctly. `load_parser` executes the source locally; use this evaluator
with the reviewed, handwritten code shown here.

In [ ]:
CASES = (
    ("30s", 30),
    ("90s", 90),
    ("2m", 120),
    ("3m", 180),
    ("1h", 3600),
    ("2h", 7200),
)


def load_parser(source):
    namespace = {}
    exec(source, namespace)
    return namespace["parse_seconds"]


def evaluate(source):
    parse = load_parser(source)
    passed = 0
    for text, expected in CASES:
        actual = parse(text)
        if type(actual) is int and actual == expected:
            passed += 1
    return passed / len(CASES)


print(f"Starting score: {evaluate(SEED):.0%}")
# Output:
# Starting score: 33%

**Revisions.** The simulated agent adds minutes support, then hours support.

In [ ]:
MINUTES = '''def parse_seconds(text):
    quantity = int(text[:-1])
    return quantity * 60 if text[-1] == "m" else quantity
'''

COMPLETE = '''def parse_seconds(text):
    quantity = int(text[:-1])
    seconds_per_unit = {"s": 1, "m": 60, "h": 3600}
    return quantity * seconds_per_unit[text[-1]]
'''

**Proposer.** `propose` supplies the instructions and current source to the
simulated agent. Its response depends on the source, so reruns repeat the
same revisions without a hidden response counter.

In [ ]:
def agent(message):
    """Simulate source revisions, without a model or a call counter."""
    source = message.rsplit("Current source:\n", 1)[1]
    if source == SEED:
        return MINUTES
    if source in (MINUTES, COMPLETE):
        return COMPLETE
    raise ValueError("This simulation only recognizes the tutorial sources.")


def propose(source):
    message = f"{CONTRACT}\nCurrent source:\n{source}"
    return agent(message)

The [Start here walkthrough](https://sentient-xyz.github.io/meta-evolve-docs/start-here/) introduces the starting parser.


<a id="3-allow-one-proposal-attempt"></a>

## 1. Allow one proposal attempt

[`meta.improve()`][meta_evolve.improve] evaluates the seed, then proposes and
evaluates revisions from the best version so far. `trials=1` limits it to one
proposal attempt. The seed evaluation is additional.

The returned run provides the selected measurement through `best_trial()`,
work counts through `usage()`, and its recorded stop reason through `inspect()`.

In [ ]:
allowance_result = meta.improve(
    seed=SEED,
    proposer=propose,
    evaluator=evaluate,
    trials=1,
)
print(f"Selected score: {allowance_result.best_trial().metrics['score']:.0%}")
print("Proposals:", allowance_result.usage().trials)
print("Evaluations:", allowance_result.usage().evaluations)
print("Stop:", allowance_result.inspect().overview.stop_reason)
# Output:
# Selected score: 67%
# Proposals: 1
# Evaluations: 2
# Stop: trial_limit_reached

Every proposal attempt counts, even if it fails to improve the result.
A failed proposal can consume an attempt without producing a candidate to evaluate.
That is why proposals and evaluations have separate counts.

<a id="4-stop-when-the-score-is-good-enough"></a>

## 2. Stop when the score is good enough

[`meta.Task`][meta_evolve.Task] declares the evaluator and objectives.
[`meta.Maximize("score", satisfy=0.6)`][meta_evolve.Maximize] means larger scores
are better and a score of at least `0.6` is sufficient to stop.

[`meta.Experiment`][meta_evolve.Experiment] combines the task, starting artifact,
proposer, and search policy. **Greedy** builds on the best measured version.
Its `max_trials=3` allows up to three proposals; [`meta.run()`][meta_evolve.run]
executes that declaration.

In [ ]:
threshold_experiment = meta.Experiment(
    seed=SEED,
    proposer=propose,
    task=meta.Task(
        evaluator=evaluate,
        objectives=(meta.Maximize("score", satisfy=0.6),),
    ),
    search=meta.Greedy(max_trials=3),
)

threshold_result = meta.run(threshold_experiment)
print(f"Selected score: {threshold_result.best_trial().metrics['score']:.0%}")
print("Proposals:", threshold_result.usage().trials)
print("Evaluations:", threshold_result.usage().evaluations)
print("Stop:", threshold_result.inspect().overview.stop_reason)
# Output:
# Selected score: 67%
# Proposals: 1
# Evaluations: 2
# Stop: objective_satisfied

The minutes revision passes four of six checks, exceeding the threshold. Search
stops with two proposal attempts unused. This target is optional: `improve()`
does not declare one automatically. Passing it measures these development
checks; assessing unseen inputs requires separate final checks.

<a id="5-put-a-ceiling-on-evaluation-work"></a>

## 3. Put a ceiling on evaluation work

[`meta.Budget`][meta_evolve.Budget] limits allowed work independently of search.
Here the task allows two evaluations in total, including the seed. Set the
target to 100% so we can see what happens when work runs out first.

In [ ]:
limited_experiment = meta.Experiment(
    seed=SEED,
    proposer=propose,
    task=meta.Task(
        evaluator=evaluate,
        objectives=(meta.Maximize("score", satisfy=1.0),),
        budget=meta.Budget(evaluations=2),
    ),
    search=meta.Greedy(max_trials=3),
)

limited_result = meta.run(limited_experiment)
print(f"Selected score: {limited_result.best_trial().metrics['score']:.0%}")
print("Proposals:", limited_result.usage().trials)
print("Evaluations:", limited_result.usage().evaluations)
print("Stop:", limited_result.inspect().overview.stop_reason)
# Output:
# Selected score: 67%
# Proposals: 1
# Evaluations: 2
# Stop: budget_exhausted

The seed and minutes revision use both evaluations. The task's ceiling stops
search before it asks for another revision, even though the target is unmet
and the search policy has remaining attempts. The selected result is retained.

<a id="6-narrow-the-budget-for-one-execution"></a>

## 4. Narrow the budget for one execution

Pass `budget=` to `run()` for a tighter allowance on a particular execution.
Omitted or `None` dimensions inherit task ceilings, and supplied finite values
cannot exceed them. A partial override such as `Budget(evaluations=1)` retains
every other task limit; see the
[reference example](https://sentient-xyz.github.io/meta-evolve-docs/reference/api-examples/#tighten-one-budget-dimension).
This call uses
`limited_experiment` from the preceding example but allows only the seed evaluation:

In [ ]:
seed_only = meta.run(limited_experiment, budget=meta.Budget(evaluations=1))
print(f"Selected score: {seed_only.best_trial().metrics['score']:.0%}")
print("Proposals:", seed_only.usage().trials)
print("Evaluations:", seed_only.usage().evaluations)
print("Stop:", seed_only.inspect().overview.stop_reason)
# Output:
# Selected score: 33%
# Proposals: 0
# Evaluations: 1
# Stop: budget_exhausted

Each run accounts for its own work. The seed preview in setup and earlier runs
are separate from the counts printed by this call.

## Change and predict

Change `evaluations=2` to `evaluations=3` in the
[evaluation ceiling example](#5-put-a-ceiling-on-evaluation-work) and rerun that cell.
Both revisions now fit: expect 100%, two proposals, three evaluations, and
`objective_satisfied`.

Then change `satisfy=0.6` to `satisfy=0.3` in the
[target score example](#4-stop-when-the-score-is-good-enough) and rerun that cell.
The seed already qualifies: expect 33%, zero proposals, one evaluation, and
`objective_satisfied`. Search never needs to ask the proposer for a revision.

For your own experiment, replace `SEED`, `propose`, and `evaluate` with your
starting value, revision function, and independent grading function. Keep the
same controls and inspect both usage and the stop reason: an allowance is a
ceiling, not a promise that all the work will be used.

## Other resources

For model calls, components must report their token, monetary, and elapsed-time
usage. A budget declaration does not interrupt an in-flight SDK request or
guarantee a hard provider spending cap. The [budget reference](https://sentient-xyz.github.io/meta-evolve-docs/guides/evaluation-and-budgets/)
explains enforcement and unknown billed spend. The
[budget concepts](https://sentient-xyz.github.io/meta-evolve-docs/concepts/budgets/) explain work counts and why candidates
cannot widen their own allowance.

Next, [save and reopen a run](https://sentient-xyz.github.io/meta-evolve-docs/learn/06-persist-run/) so its history and
selected source remain available after you close the notebook.